In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import re

from setting_for_sdm.path_setting import path_list
from setting_for_sdm.date_setting import Date_Setting
from setting_for_sdm.constants import CONSTANTS

import lib.stats.stats as st
from lib.utils.file_io import *
from lib.utils.statistics import *
from lib.utils.settings import set_matplotlib
from matplotlib import pyplot as plt

import matplotlib as mpl
import lib.visualization.plot_generator as PlotGen
import lib.visualization.font_setting as font_setting
mpl.rcParams['font.family'] = font_setting.init_font()

import statsmodels.formula.api as smf
from scipy import stats





Helvetica /home/mghan/.fonts/Helvetica/Helvetica Oblique.ttf
Registered font name: Helvetica


In [2]:
def draw_cognitive_complexity_scatter_plot(idx_num, viz_df, option_dict, output_dir, opt):
    sharey = False ## 또는 sharey=False
    sharex = True ## 또는 sharex=False

    fig, axs = mpl.pyplot.subplots(figsize = (12, 6), constrained_layout=True)

    # st_chow_1year = st.Stats(x, y,  0.95, viz_df[viz_df['rel_week'] ==0].index.values[0])
    # F_stat_1, p_value_1 = st_chow_1year.chow_test()
    plotgen = PlotGen.PlotGen()
    x= list(viz_df['rel_week'])
    y= list(viz_df['Cognitive Complexity'])
    
    plotgen.draw_regression_with_chow(axs
                                , f'Cognitive Complexity for {option_dict["selected_tags"]}({opt})' 
                                , x
                                , y
                                , None
                                , None
                                , None
                                , False
                                )

    fig.supxlabel("Weeks relative to ChatGPT release", fontsize=22) 
    fig.supylabel('Weekly average complexity', fontsize=22)
    filename = f"{idx_num}_scatter_plot_{option_dict['selected_tags']}_cognitive_complexity_{option_dict['year_range']}_{opt}.png"
    plt.savefig(
    os.path.join(output_dir, filename),
    dpi=300,
    bbox_inches='tight'
    )
    plt.close(fig)

In [2]:
def filter_df_with_tag(df, tags):
    pattern = '|'.join(re.escape(f'<{tag}>') for tag in tags)

    mask = df['tags'].astype(str).str.contains(pattern, regex=True)
    return df[mask]

In [9]:
def did_top_vs_bottom(df_top, df_bot, lang):
    df_top = df_top.copy()
    df_bot = df_bot.copy()
    df_top['treated'] = 1
    df_bot['treated'] = 0
    
    df_combined = pd.concat([df_top, df_bot], axis=0)
    df_combined['post'] = (df_combined['rel_week'] >= 0).astype(int)
    
    model = smf.ols(
        'complexity ~ rel_week + post + treated + rel_week:post + rel_week:treated + post:treated + rel_week:post:treated',
        data=df_combined
    ).fit()
    
    # rel_week:post:treated 가 DID 효과
    effect = model.params['rel_week:post:treated']
    se = model.bse['rel_week:post:treated']
    pval = model.pvalues['rel_week:post:treated']
    
    return {
        'language': lang,
        'effect': effect,
        'se': se,
        'pvalue': pval,
        'ci_lower': effect - 1.96 * se,
        'ci_upper': effect + 1.96 * se,
        'n': len(df_combined),
        'significant': pval < 0.05
    }

In [10]:
def comprehensive_meta_analysis_from_df(df_results, tag_group_name):
    """
    이미 계산된 individual DID 결과로 메타분석 수행
    
    Args:
        df_results: DataFrame with columns ['language', 'effect', 'se', 'pvalue', 
                    'ci_lower', 'ci_upper', 'n', 'significant']
        tag_group_name: str, 분석 그룹 이름 (e.g., 'Top 20% vs Bottom 20% DID')
    
    Returns:
        dict with meta-analysis results
    """
    df_results = df_results.copy()
    df_results['var'] = df_results['se'] ** 2
    
    print(f"\n{'='*70}")
    print(f"{tag_group_name} - META-ANALYSIS RESULTS")
    print(f"{'='*70}\n")
    
    # ── Step 1: Fixed-effect model ──
    weights = 1 / df_results['var']
    fixed_effect = np.sum(df_results['effect'] * weights) / np.sum(weights)
    fixed_se = np.sqrt(1 / np.sum(weights))
    fixed_z = fixed_effect / fixed_se
    fixed_p = 2 * (1 - stats.norm.cdf(abs(fixed_z)))
    
    print("FIXED-EFFECT MODEL:")
    print(f"  Pooled effect: {fixed_effect:.6f} (SE: {fixed_se:.6f})")
    print(f"  95% CI: [{fixed_effect - 1.96*fixed_se:.6f}, {fixed_effect + 1.96*fixed_se:.6f}]")
    print(f"  Z = {fixed_z:.3f}, p = {fixed_p:.4f}")
    
    # ── Step 2: Heterogeneity ──
    Q = np.sum(weights * (df_results['effect'] - fixed_effect)**2)
    df_q = len(df_results) - 1
    Q_pval = 1 - stats.chi2.cdf(Q, df_q) if df_q > 0 else 1.0
    I2 = max(0, 100 * (Q - df_q) / Q) if Q > 0 else 0
    
    if I2 < 25:
        het_interp = "low"
    elif I2 < 50:
        het_interp = "moderate"
    elif I2 < 75:
        het_interp = "substantial"
    else:
        het_interp = "considerable"
    
    print(f"\nHETEROGENEITY:")
    print(f"  Cochran's Q = {Q:.2f} (df = {df_q}, p = {Q_pval:.4f})")
    print(f"  I² = {I2:.1f}% → {het_interp} heterogeneity")
    
    # ── Step 3: Random-effects model (DerSimonian-Laird) ──
    C = np.sum(weights) - np.sum(weights**2) / np.sum(weights)
    tau2 = max(0, (Q - df_q) / C) if C > 0 else 0
    
    random_weights = 1 / (df_results['var'] + tau2)
    random_effect = np.sum(df_results['effect'] * random_weights) / np.sum(random_weights)
    random_se = np.sqrt(1 / np.sum(random_weights))
    random_z = random_effect / random_se
    random_p = 2 * (1 - stats.norm.cdf(abs(random_z)))
    
    print(f"\nRANDOM-EFFECTS MODEL:")
    print(f"  τ² = {tau2:.6f}")
    print(f"  Pooled effect: {random_effect:.6f} (SE: {random_se:.6f})")
    print(f"  95% CI: [{random_effect - 1.96*random_se:.6f}, {random_effect + 1.96*random_se:.6f}]")
    print(f"  Z = {random_z:.3f}, p = {random_p:.4f}")
    
    # ── Step 4: Subgroup analysis ──
    sig_langs = df_results[df_results['significant'] == True]
    nonsig_langs = df_results[df_results['significant'] == False]
    
    print(f"\nSUBGROUP ANALYSIS:")
    print(f"  Significant languages (p<0.05): {len(sig_langs)}/{len(df_results)}")
    if len(sig_langs) > 0:
        print(f"    Mean effect: {sig_langs['effect'].mean():.6f}")
        print(f"    Languages: {', '.join(sig_langs['language'].tolist())}")
    print(f"  Non-significant languages: {len(nonsig_langs)}/{len(df_results)}")
    if len(nonsig_langs) > 0:
        print(f"    Mean effect: {nonsig_langs['effect'].mean():.6f}")
    
    # ── Step 5: Egger's test (publication bias) ──
    if len(df_results) >= 10:
        precision = 1 / df_results['se']
        standard_effect = df_results['effect'] / df_results['se']
        
        egger_model = smf.ols(
            'standard_effect ~ precision',
            data=pd.DataFrame({
                'standard_effect': standard_effect,
                'precision': precision
            })
        ).fit()
        
        egger_intercept = egger_model.params['Intercept']
        egger_p = egger_model.pvalues['Intercept']
        
        print(f"\nEGGER'S TEST (Publication Bias):")
        print(f"  Intercept = {egger_intercept:.4f}, p = {egger_p:.4f}")
        if egger_p < 0.05:
            print(f"  → Evidence of publication bias/small-study effects")
        else:
            print(f"  → No strong evidence of publication bias")
    else:
        egger_intercept = None
        egger_p = None
        print(f"\nEGGER'S TEST: Skipped (need ≥ 10 studies, have {len(df_results)})")
    
    return {
        'individual': df_results,
        'fixed_effect': fixed_effect,
        'fixed_se': fixed_se,
        'fixed_p': fixed_p,
        'random_effect': random_effect,
        'random_se': random_se,
        'random_p': random_p,
        'tau2': tau2,
        'Q': Q,
        'Q_pval': Q_pval,
        'I2': I2,
        'n_languages': len(df_results),
        'n_significant': len(sig_langs)
    }

In [ ]:
did_results = []

run_id_start = 10000
for idx, lang in enumerate(CONSTANTS.src_extend.keys()):
    print(f'[visualizing....] start visualizing {lang} language')
    top_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_top_tags.json')
    bot_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_bot_tags.json')
    viz_dir = f'{path_list["data_root_dir"]}/result/code_complexity/run_id_{run_id_start+idx}'
    data_dir = f"{viz_dir}/data/csv"
    option_dict = load_json(f"{viz_dir}/data/option.json")

    
    output_dir = create_dir('./fig/')
    date_range = 'Weekly'
    std_date = Date_Setting[option_dict['year_range']]['std_date']

    df = pd.read_parquet(f'{option_dict["save_dir"]}/data/all_complexity.parquet')
    df['id'] = df['Path'].apply(lambda x : x.split('_')[1].split('.')[0])
    df[['id', 'Cognitive Complexity']] = df[['id', 'Cognitive Complexity']].astype(int)
    df = df.groupby('id', as_index=False)['Cognitive Complexity'].sum()

    origin_df = load_df(option_dict['data_dir'], ['id', 'creationdate', 'title','tags', 'body'])
    filtered_top_df = filter_df_with_tag(origin_df, top_tags)
    filtered_bot_df = filter_df_with_tag(origin_df, bot_tags)


    top_viz_df = pd.merge(df, filtered_top_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]
    bot_viz_df = pd.merge(df, filtered_bot_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]

    top_viz_df['rel_week'] = np.floor((pd.to_datetime(top_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    top_viz_df = (top_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())
    top_viz_df.rename(columns={'Cognitive Complexity': 'complexity'}, inplace=True)

    bot_viz_df['rel_week'] = np.floor((pd.to_datetime(bot_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    bot_viz_df = (bot_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())
    bot_viz_df.rename(columns={'Cognitive Complexity': 'complexity'}, inplace=True)

    try:
        result = did_top_vs_bottom(top_viz_df, bot_viz_df, lang)
        did_results.append(result)
        print(f'  effect={result["effect"]:.6f}, p={result["pvalue"]:.4f}')
    except Exception as e:
        print(f'  Skipped: {e}')
    
    print(f'[End....] {lang}')

# 결과를 DataFrame으로
df_did_results = pd.DataFrame(did_results)

# 메타분석 실행
meta_result = comprehensive_meta_analysis_from_df(df_did_results, 'Top 20% vs Bottom 20% DID')

[visualizing....] start visualizing python language


100%|██████████| 65/65 [00:17<00:00,  3.75it/s]


  effect=-0.005720, p=0.6558
[End....] python
[visualizing....] start visualizing javascript language


100%|██████████| 65/65 [00:16<00:00,  3.87it/s]


  effect=-0.017243, p=0.1221
[End....] javascript
[visualizing....] start visualizing java language


...

  effect=0.003384, p=0.6668
[End....] java
[visualizing....] start visualizing c# language


100%|██████████| 65/65 [00:08<00:00,  7.63it/s]


  effect=-0.019254, p=0.0745
[End....] c#
[visualizing....] start visualizing c++ language


100%|██████████| 65/65 [00:05<00:00, 11.79it/s]


KeyboardInterrupt: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch


def publication_forest_plot(meta_results, title, save_path=None):
    """
    Science Advances style forest plot for meta-analysis results.
    """
    # ---- Science Advances-ish rcParams ----
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 8,
        "axes.labelsize": 9,
        "axes.titlesize": 9.5,
        "xtick.labelsize": 7.5,
        "ytick.labelsize": 8,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # color palette (matches the language forest plot)
    C_POS = "#C0392B"   # significant & positive-ish (warm red)
    C_NEG = "#2C5F8D"   # significant & negative / non-sig cool
    C_NS  = "#8A8A8A"   # non-significant (muted gray)
    C_POOL = "#1F4E79"  # pooled diamond (deep blue)
    C_REF = "0.55"
    C_GRID = "0.92"

    # ---- data prep ----
    df = meta_results["individual"].copy()
    order_map = {lang: i for i, lang in enumerate(CONSTANTS.languages_from2020to2022)}
    df = df.sort_values(by="language", key=lambda x: x.map(order_map), ascending=False).reset_index(drop=True)

    n = len(df)
    y_pos = np.arange(n)

    # point sizes based on inverse-variance weights
    weights = 1.0 / (df["var"] + meta_results["tau2"])
    w_norm = weights / weights.max()
    smin, smax = 14, 80
    sizes = smin + (smax - smin) * np.sqrt(w_norm)

    # colors by significance and sign
    def _color(pval, eff):
        if pval >= 0.05:
            return C_NS
        return C_POS if eff > 0 else C_NEG
    colors = [_color(r.pvalue, r.effect) for r in df.itertuples()]

    # ---- figure with explicit margins ----
    fig_height = max(6.5, 0.30 * n + 2.6)
    fig = plt.figure(figsize=(6.4, fig_height))
    # [left, bottom, width, height] — no right-side number column anymore
    ax = fig.add_axes([0.22, 0.13, 0.74, 0.78])

    # ---- zero reference line ----
    ax.axvline(0, color=C_REF, lw=0.6, ls="--", zorder=0)

    # ---- CI bars ----
    for i, row in enumerate(df.itertuples()):
        ax.plot([row.ci_lower, row.ci_upper], [i, i],
                color=colors[i], lw=1.0, alpha=0.85,
                solid_capstyle="round", zorder=2)

    # ---- points ----
    ax.scatter(df["effect"], y_pos, s=sizes, c=colors,
               edgecolor="white", linewidth=0.5, zorder=3)

    # ---- pooled effect (diamond) ----
    pooled = meta_results["random_effect"]
    pooled_se = meta_results["random_se"]
    pooled_lo = pooled - 1.96 * pooled_se
    pooled_hi = pooled + 1.96 * pooled_se

    diamond_y = n + 1.0
    diamond_half = 0.35
    diamond_x = [pooled_lo, pooled, pooled_hi, pooled]
    diamond_y_coords = [diamond_y, diamond_y + diamond_half,
                        diamond_y, diamond_y - diamond_half]
    ax.fill(diamond_x, diamond_y_coords, color=C_POOL,
            alpha=0.9, edgecolor="white", linewidth=0.8, zorder=4)

    # ---- y-axis ----
    ytick_labels = list(df["language"]) + ["Pooled (RE)"]
    ytick_positions = list(y_pos) + [diamond_y]
    ax.set_yticks(ytick_positions)
    ax.set_yticklabels(ytick_labels)
    ax.get_yticklabels()[-1].set_fontweight("bold")
    ax.set_ylim(-0.8, diamond_y + 1.0)

    # ---- x-axis ----
    ax.set_xlabel("DID effect (slope change per week)", labelpad=6)

    # ---- title ----
    ax.set_title(title, loc="left", fontweight="bold", pad=12)

    # ---- style ----
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.grid(axis="x", lw=0.4, color=C_GRID, zorder=0)
    ax.set_axisbelow(True)

    # ---- legend (upper-left inside the plot) ----
    legend_elements = [
        Patch(facecolor=C_POS, label="p < 0.05, positive"),
        Patch(facecolor=C_NEG, label="p < 0.05, negative"),
        Patch(facecolor=C_NS,  label="p ≥ 0.05"),
        Patch(facecolor=C_POOL, label="Pooled (random-effects)"),
    ]
    ax.legend(handles=legend_elements, loc="upper left",
              bbox_to_anchor=(0.01, 0.99),
              fontsize=6.8, labelspacing=0.6,
              borderpad=0.6, handletextpad=0.6)

    # ---- caption (with heterogeneity statistics) ----
    caption = (
        f"Random-effects meta-analysis across {n} languages.  "
        f"Heterogeneity: I² = {meta_results['I2']:.1f}%, "
        f"τ² = {meta_results['tau2']:.4f}, "
        f"Q({n-1}) = {meta_results['Q']:.2f}, "
        f"p = {meta_results['Q_pval']:.4f}.  "
        f"Error bars: 95% CI.  Point size ∝ √(inverse-variance weight).  "
        f"Blue diamond: pooled random-effects estimate."
    )
    fig.text(0.04, 0.015, caption,
             fontsize=6.2, color="0.35",
             ha="left", va="bottom", wrap=True)

    # ---- save ----
    if save_path:
        fig.savefig(save_path, dpi=600, bbox_inches=None)
        if save_path.endswith(".pdf"):
            fig.savefig(save_path.replace(".pdf", ".png"), dpi=600, bbox_inches=None)
            fig.savefig(save_path.replace(".pdf", ".svg"), bbox_inches=None)

    return fig

In [ ]:
# 변경: 1개 플롯
fig = publication_forest_plot(
    meta_result, 
    'Top 20% vs Bottom 20% Tags (DID)', 
    'forest_did_top_vs_bottom.png'
)
plt.show()

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf


# =============================================================
# 1) 언어 하나에 대한 DID 분석
# =============================================================
def did_top_vs_bottom(df_top, df_bot, lang):
    """
    한 언어 안에서 top 20% 태그 vs bottom 20% 태그의
    추세 변화 차이를 triple-interaction DID로 추정.
    
    rel_week:post:treated 계수 = 기준일 이후 상위 태그의 기울기 변화가
                                  하위 태그의 기울기 변화보다 얼마나 더/덜 큰가
    """
    df_top = df_top.copy()
    df_bot = df_bot.copy()
    df_top["treated"] = 1
    df_bot["treated"] = 0

    df_combined = pd.concat([df_top, df_bot], axis=0)
    df_combined["post"] = (df_combined["rel_week"] >= 0).astype(int)

    model = smf.ols(
        "complexity ~ rel_week + post + treated "
        "+ rel_week:post + rel_week:treated + post:treated "
        "+ rel_week:post:treated",
        data=df_combined,
    ).fit()

    key = "rel_week:post:treated"
    effect = model.params[key]
    se = model.bse[key]
    pval = model.pvalues[key]

    return {
        "language": lang,
        "effect": effect,
        "se": se,
        "pvalue": pval,
        "ci_lower": effect - 1.96 * se,
        "ci_upper": effect + 1.96 * se,
        "n": len(df_combined),
        "significant": pval < 0.05,
    }


# =============================================================
# 2) 언어별 DID 반복 실행
# =============================================================
def run_per_language_did(languages):
    """각 언어에 대해 DID를 돌리고 결과 DataFrame 반환."""
    results = []
    run_id_start = 10000
    for idx, lang in enumerate(CONSTANTS.src_extend.keys()):
        print(f'[visualizing....] start visualizing {lang} language')
        top_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_top_tags.json')
        bot_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_bot_tags.json')
        viz_dir = f'{path_list["data_root_dir"]}/result/code_complexity/run_id_{run_id_start+idx}'
        data_dir = f"{viz_dir}/data/csv"
        option_dict = load_json(f"{viz_dir}/data/option.json")

        
        output_dir = create_dir('./fig/')
        date_range = 'Weekly'
        std_date = Date_Setting[option_dict['year_range']]['std_date']

        df = pd.read_parquet(f'{option_dict["save_dir"]}/data/all_complexity.parquet')
        df['id'] = df['Path'].apply(lambda x : x.split('_')[1].split('.')[0])
        df[['id', 'Cognitive Complexity']] = df[['id', 'Cognitive Complexity']].astype(int)
        df = df.groupby('id', as_index=False)['Cognitive Complexity'].sum()

        origin_df = load_df(option_dict['data_dir'], ['id', 'creationdate', 'title','tags', 'body'])
        filtered_top_df = filter_df_with_tag(origin_df, top_tags)
        filtered_bot_df = filter_df_with_tag(origin_df, bot_tags)


        top_viz_df = pd.merge(df, filtered_top_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]
        bot_viz_df = pd.merge(df, filtered_bot_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]

        top_viz_df['rel_week'] = np.floor((pd.to_datetime(top_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
        top_viz_df = (top_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())
        top_viz_df.rename(columns={'Cognitive Complexity': 'complexity'}, inplace=True)

        bot_viz_df['rel_week'] = np.floor((pd.to_datetime(bot_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
        bot_viz_df = (bot_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())
        bot_viz_df.rename(columns={'Cognitive Complexity': 'complexity'}, inplace=True)


        

        try:
            result = did_top_vs_bottom(top_viz_df, bot_viz_df, lang)
            results.append(result)
            print(f"  effect={result['effect']:.6f}, "
                  f"p={result['pvalue']:.4f}")
        except Exception as e:
            print(f"  Skipped: {e}")
        print(f"[End] {lang}\n")

    return pd.DataFrame(results)


# =============================================================
# 3) 결과 테이블 보기 좋게 출력
# =============================================================
def print_results_table(df_results):
    """언어별 DID 결과를 보기 좋게 출력."""
    print("\n" + "=" * 78)
    print("PER-LANGUAGE DID RESULTS")
    print("=" * 78)
    print(f"{'Language':<20} {'Effect':>10} {'95% CI':>24} "
          f"{'p-value':>10} {'sig':>5}")
    print("-" * 78)

    # effect 기준 정렬 (가장 강한 음의 효과부터)
    df_sorted = df_results.sort_values("effect")
    order_map = {lang: i for i, lang in enumerate(CONSTANTS.src_extend.keys())}
    df = df.sort_values(
            by="language",
            key=lambda x: x.map(order_map),
            ascending=False,
        ).reset_index(drop=True)
    
    for _, row in df_sorted.iterrows():
        ci_str = f"[{row['ci_lower']:+.4f}, {row['ci_upper']:+.4f}]"
        sig = "★" if row["significant"] else ""
        print(f"{row['language']:<20} {row['effect']:+10.6f} "
              f"{ci_str:>24} {row['pvalue']:>10.4f} {sig:>5}")
    print("-" * 78)
    print(f"Total: {len(df_results)} languages, "
          f"{df_results['significant'].sum()} significant (p<0.05)")


# =============================================================
# 4) Forest plot (핵심 시각화)
# =============================================================
def did_forest_plot(df_results, title, 
                     order_list=None, save_path=None):
    """
    Science Advances style forest plot for per-language DID results.
    
    Args:
        df_results: DataFrame with columns 
                    [language, effect, se, pvalue, ci_lower, ci_upper, significant]
        title: plot title
        order_list: optional list of language names defining display order
                    (top to bottom); defaults to effect-sorted
        save_path: output path (png/pdf/svg)
    """
    # ---- Science Advances-ish rcParams ----
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["DejaVu Sans"],
        "font.size": 8,
        "axes.labelsize": 9,
        "axes.titlesize": 9.5,
        "xtick.labelsize": 7.5,
        "ytick.labelsize": 8,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

    # ---- color palette (matched to publication_forest_plot) ----
    C_POS = "#C0392B"   # significant & positive (warm red)
    C_NEG = "#2C5F8D"   # significant & negative (cool blue)
    C_NS  = "#8A8A8A"   # non-significant (muted gray)
    C_REF = "0.55"
    C_GRID = "0.92"

    # ---- data prep ----
    df = df_results.copy()
    
    if order_list is not None:
        order_map = {lang: i for i, lang in enumerate(order_list)}
        df = df.sort_values(
            by="language",
            key=lambda x: x.map(order_map),
            ascending=False,
        ).reset_index(drop=True)
    else:
        df = df.sort_values("effect", ascending=False).reset_index(drop=True)

    n = len(df)
    y_pos = np.arange(n)

    # point sizes based on inverse-variance (1 / se^2)
    var = df["se"] ** 2
    weights = 1.0 / var
    w_norm = weights / weights.max()
    smin, smax = 14, 80
    sizes = smin + (smax - smin) * np.sqrt(w_norm)

    # colors by significance and sign
    def _color(pval, eff):
        if pval >= 0.05:
            return C_NS
        return C_POS if eff > 0 else C_NEG

    colors = [_color(r.pvalue, r.effect) for r in df.itertuples()]

    # ---- figure with explicit margins ----
    fig_height = max(6.5, 0.30 * n + 2.6)
    fig = plt.figure(figsize=(6.4, fig_height))
    ax = fig.add_axes([0.22, 0.13, 0.74, 0.78])

    # ---- zero reference line ----
    ax.axvline(0, color=C_REF, lw=0.6, ls="--", zorder=0)

    # ---- CI bars ----
    for i, row in enumerate(df.itertuples()):
        ax.plot(
            [row.ci_lower, row.ci_upper], [i, i],
            color=colors[i], lw=1.0, alpha=0.85,
            solid_capstyle="round", zorder=2,
        )

    # ---- points ----
    ax.scatter(
        df["effect"], y_pos,
        s=sizes, c=colors,
        edgecolor="white", linewidth=0.5, zorder=3,
    )

    # ---- y-axis ----
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df["language"])
    ax.set_ylim(-0.8, n - 0.2)

    # ---- x-axis ----
    ax.set_xlabel("DID effect (slope change per week)", labelpad=6)

    # ---- title ----
    ax.set_title(title, loc="left", fontweight="bold", pad=12)

    # ---- style ----
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.grid(axis="x", lw=0.4, color=C_GRID, zorder=0)
    ax.set_axisbelow(True)

    # ---- legend (upper-left inside the plot) ----
    legend_elements = [
        Patch(facecolor=C_POS, label="p < 0.05, positive"),
        Patch(facecolor=C_NEG, label="p < 0.05, negative"),
        Patch(facecolor=C_NS,  label="p ≥ 0.05"),
    ]
    ax.legend(
        handles=legend_elements, loc="upper left",
        bbox_to_anchor=(0.01, 0.99),
        fontsize=6.8, labelspacing=0.6,
        borderpad=0.6, handletextpad=0.6,
    )

    # ---- caption (no meta-analysis stats) ----
    n_sig = int(df["significant"].sum())
    caption = (
        f"Per-language DID analysis across {n} languages "
        f"({n_sig} significant at p < 0.05).  "
        f"Error bars: 95% CI.  Point size ∝ √(inverse-variance weight).  "
        f"Outcome: slope change of top-20% tags relative to bottom-20% tags "
        f"after the reference date."
    )
    fig.text(
        0.04, 0.015, caption,
        fontsize=6.2, color="0.35",
        ha="left", va="bottom", wrap=True,
    )

    # ---- save ----
    if save_path:
        fig.savefig(save_path, dpi=600, bbox_inches=None)
        if save_path.endswith(".pdf"):
            fig.savefig(
                save_path.replace(".pdf", ".png"),
                dpi=600, bbox_inches=None,
            )
            fig.savefig(
                save_path.replace(".pdf", ".svg"),
                bbox_inches=None,
            )

    return fig


# =============================================================
# 사용 예
# =============================================================
# df_did_results는 앞서 run_per_language_did()로 얻은 결과
# fig = did_forest_plot(
#     df_did_results,
#     title="Per-language DID: Top 20% vs Bottom 20% tags",
#     order_list=CONSTANTS.languages_from2020to2022,  # 선택: 고정 순서
#     save_path="did_forest_plot.pdf",
# )
# plt.show()


# =============================================================
# 5) 실행
# =============================================================
df_did_results = run_per_language_did(CONSTANTS.src_extend.keys())

# 결과 테이블 출력
print_results_table(df_did_results)

# Forest plot
did_forest_plot(
    df_did_results,
    title="Per-language DID: Top 20% vs Bottom 20% tags",
    save_path="did_forest_plot.png",
)

[visualizing....] start visualizing python language


100%|██████████| 65/65 [00:13<00:00,  4.80it/s]


  effect=0.014678, p=0.5656
[End] python

[visualizing....] start visualizing javascript language


100%|██████████| 65/65 [00:09<00:00,  6.73it/s]


  effect=-0.050832, p=0.0557
[End] javascript

[visualizing....] start visualizing java language


...

100%|██████████| 65/65 [00:00<00:00, 1426.35it/s]

  effect=-0.055162, p=0.2083
[End] f#

[visualizing....] start visualizing solidity language



100%|██████████| 65/65 [00:00<00:00, 634.94it/s]


  effect=0.081981, p=0.4320
[End] solidity


PER-LANGUAGE DID RESULTS
Language                 Effect                   95% CI    p-value   sig
------------------------------------------------------------------------------


UnboundLocalError: local variable 'df' referenced before assignment

In [4]:
lang_list = list(CONSTANTS.src_extend.keys())

In [ ]:
run_id_start = 10000
for idx, lang in enumerate(lang_list[:10]):
    print(f'[visualizing....] start visualizing {lang} language')
    top_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_top_tags.json')
    bot_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_bot_tags.json')
    viz_dir = f'{path_list["data_root_dir"]}/result/code_complexity/run_id_{run_id_start+idx}'
    data_dir = f"{viz_dir}/data/csv"
    option_dict = load_json(f"{viz_dir}/data/option.json")

    
    output_dir = create_dir('./fig/')
    date_range = 'Weekly'
    std_date = Date_Setting[option_dict['year_range']]['std_date']

    df = pd.read_parquet(f'{option_dict["save_dir"]}/data/all_complexity.parquet')
    df['id'] = df['Path'].apply(lambda x : x.split('_')[1].split('.')[0])
    df[['id', 'Cognitive Complexity']] = df[['id', 'Cognitive Complexity']].astype(int)
    # df = df.groupby('id', as_index=False)['Cognitive Complexity'].max()

    origin_df = load_df(option_dict['data_dir'], ['id', 'creationdate', 'title','tags', 'body'])
    filtered_top_df = filter_df_with_tag(origin_df, top_tags)
    filtered_bot_df = filter_df_with_tag(origin_df, bot_tags)


    top_viz_df = pd.merge(df, filtered_top_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]
    bot_viz_df = pd.merge(df, filtered_bot_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]

    top_viz_df['rel_week'] = np.floor((pd.to_datetime(top_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    top_viz_df = (top_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())

    bot_viz_df['rel_week'] = np.floor((pd.to_datetime(bot_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    bot_viz_df = (bot_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())
        
    draw_cognitive_complexity_scatter_plot(run_id_start+idx, top_viz_df, option_dict, output_dir, 'top 20%')
    draw_cognitive_complexity_scatter_plot(run_id_start+idx, bot_viz_df, option_dict, output_dir, 'bot 20%')
    print('[visualizing....] end visualizing {lang} language')
    


In [5]:
run_id_start = 10000
for idx, lang in enumerate(lang_list[:10]):
    print(f'[visualizing....] start visualizing {lang} language')
    top_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_top_tags.json')
    bot_tags = load_json(f'/mnt/hdd/mghan/so_difficultyXavailability/data/{lang}_bot_tags.json')
    viz_dir = f'{path_list["data_root_dir"]}/result/code_complexity/run_id_{run_id_start+idx}'
    data_dir = f"{viz_dir}/data/csv"
    option_dict = load_json(f"{viz_dir}/data/option.json")

    
    output_dir = create_dir('./fig/')
    date_range = 'Weekly'
    std_date = Date_Setting[option_dict['year_range']]['std_date']

    df = pd.read_parquet(f'{option_dict["save_dir"]}/data/all_complexity.parquet')
    df['id'] = df['Path'].apply(lambda x : x.split('_')[1].split('.')[0])
    df[['id', 'Cognitive Complexity']] = df[['id', 'Cognitive Complexity']].astype(int)
    # df = df.groupby('id', as_index=False)['Cognitive Complexity'].max()

    origin_df = load_df(option_dict['data_dir'], ['id', 'creationdate', 'title','tags', 'body'])
    filtered_top_df = filter_df_with_tag(origin_df, top_tags)
    filtered_bot_df = filter_df_with_tag(origin_df, bot_tags)


    top_viz_df = pd.merge(df, filtered_top_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]
    bot_viz_df = pd.merge(df, filtered_bot_df, on = 'id')[['id', 'creationdate', 'Cognitive Complexity']]

    top_viz_df['rel_week'] = np.floor((pd.to_datetime(top_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    top_viz_df = (top_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())

    bot_viz_df['rel_week'] = np.floor((pd.to_datetime(bot_viz_df['creationdate'], format='mixed')- std_date).dt.days/7)
    bot_viz_df = (bot_viz_df.groupby('rel_week', as_index=False)['Cognitive Complexity'].mean())
        
    draw_cognitive_complexity_scatter_plot(run_id_start+idx, top_viz_df, option_dict, output_dir, 'top 20%')
    draw_cognitive_complexity_scatter_plot(run_id_start+idx, bot_viz_df, option_dict, output_dir, 'bot 20%')
    print('[visualizing....] end visualizing {lang} language')
    


[visualizing....] start visualizing python language


100%|██████████| 65/65 [00:13<00:00,  4.77it/s]


[visualizing....] end visualizing {lang} language
[visualizing....] start visualizing javascript language


100%|██████████| 65/65 [00:10<00:00,  6.36it/s]


[visualizing....] end visualizing {lang} language
[visualizing....] start visualizing java language


...

[visualizing....] end visualizing {lang} language
[visualizing....] start visualizing swift language


100%|██████████| 65/65 [00:00<00:00, 68.71it/s]


[visualizing....] end visualizing {lang} language
[visualizing....] start visualizing kotlin language


100%|██████████| 65/65 [00:00<00:00, 75.23it/s]


[visualizing....] end visualizing {lang} language
